# 09 — ITER Disruption Classifier

11 physics-based ITER features mapped to a quantum circuit classifier.
Synthetic data generation, normalization, and benchmark evaluation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from scpn_quantum_control.control.q_disruption_iter import (
    DisruptionBenchmark,
    ITERFeatureSpec,
    generate_synthetic_iter_data,
)

## 1. ITER Feature Space

In [ ]:
spec = ITERFeatureSpec()
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(spec.names))
width = 0.35
ax.bar(x - width / 2, spec.mins, width, label="Min", color="#1f77b4")
ax.bar(x + width / 2, spec.maxs, width, label="Max", color="#d62728")
ax.set_xticks(x)
ax.set_xticklabels(spec.names, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Value")
ax.set_title("ITER Disruption Feature Ranges (Nuclear Fusion 39, 1999)")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 2. Synthetic Data

In [ ]:
X, y = generate_synthetic_iter_data(500, disruption_fraction=0.3, rng=np.random.default_rng(42))
print(f"Samples: {X.shape[0]}, Features: {X.shape[1]}")
print(f"Disruptions: {int(y.sum())} ({y.mean() * 100:.0f}%)")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, feat_idx, name in zip(axes, [1, 4, 6], ["q95", "beta_N", "locked_mode"]):
    safe = X[y == 0, feat_idx]
    disrupt = X[y == 1, feat_idx]
    ax.hist(safe, bins=20, alpha=0.6, label="Safe", color="#2ca02c")
    ax.hist(disrupt, bins=20, alpha=0.6, label="Disruption", color="#d62728")
    ax.set_title(name)
    ax.legend(fontsize=8)
plt.suptitle("Key Disruption Discriminators (normalized)", fontsize=12)
plt.tight_layout()
plt.show()

## 3. Quantum Classifier Benchmark

In [ ]:
bench = DisruptionBenchmark(n_train=80, n_test=40, seed=42)
result = bench.run(epochs=5, lr=0.1)
print(f"Accuracy: {result['accuracy']:.1%}")
print(f"Train: {result['n_train']}, Test: {result['n_test']}")